# Build 03-04 · Prediction datasets — file-level diagnostics of the mitigated scores

**Kernel: the analysis `.venv`** (`python3`) — parquet only: no model, no subprocess, and
deliberately **no labels**. 03_03 wrote the score files right after retraining; this notebook
answers *are those files sound, and what do the score distributions say* — before any labelled
evaluation:

| § | question |
|---|---|
| 2 | inventory & integrity — every (axis, split) file present? full row coverage vs the split's targets? NaNs, range, duplicate ids? |
| 3 | distributions — summary quantiles + ECDF overlay per split |
| 4 | cutoff mass — share above the production cutoff(s) and in the boundary band: the label-free *would-scrap volume* proxy |
| 5 | agreement — how far each axis moved from baseline and from each other (mean/max |Δ|, Spearman) |

Labelled evaluation (precision/recall, decisions, τ tuning) is **03_05**'s job. The production
cutoffs here are reference marks only — a retrained model's score scale is its own.

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC
VERSION = "v3"
TRAIN_SPLIT = "train"
SPLITS = list(dict.fromkeys([TRAIN_SPLIT, config.OOT_SPLIT[VERSION]]))
ID_COL = "claim_id"
SCORE_COL = "model_" + VERSION + "_score"
BAND_H = 0.01
REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval"

# reference cutoffs from the production rule — marks on the score axis, nothing is decided here
_rule = config.DECISION_RULES[VERSION]
if _rule["shape"] == "global":
    TAUS = [float(_rule["threshold"])]
elif _rule["shape"] == "piecewise_global":
    TAUS = sorted({float(r["threshold"]) for r in _rule["regimes"]})
else:
    raise SystemExit("segmented rule (v1) is out of scope here")
print("splits:", SPLITS, "| reference cutoffs:", TAUS)

In [ ]:
# §2 — inventory & integrity, then one frame per split (claim_id + a column per run)
def discover(split: str) -> dict:
    """{run tag: score file} for one split — baseline (scores kind) first when it exists."""
    out = {}
    b = config.path("scores", VERSION, split=split)
    if b.is_file():
        out["baseline"] = b
    else:
        print(f"  note: no baseline scores at {b} (run src/scoring/score_all.py) — skipped")
    prefix = f"{VERSION}_mitigated_scores_{split}_"
    for p in sorted(REEVAL_DIR.glob(prefix + "*.parquet")):
        out[p.stem[len(prefix):]] = p
    return out


FILES = {sp: discover(sp) for sp in SPLITS}
assert any(t != "baseline" for sp in SPLITS for t in FILES[sp]), \
    "no mitigated score files found — run 03_03 first"

frames: dict[str, pd.DataFrame] = {}
MODELS: dict[str, list[str]] = {}
rows = []
for sp in SPLITS:
    base = pd.read_parquet(config.split_path("targets", VERSION, sp), columns=[ID_COL])
    f = base.copy()
    for tag, p in FILES[sp].items():
        s = pd.read_parquet(p)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: tag})
        f = f.merge(s, on=ID_COL, how="left", validate="one_to_one")   # raises on duplicate ids
        col = f[tag]
        rows.append({"split": sp, "run": tag, "n_scored": int(col.notna().sum()),
                     "n_targets": len(base), "coverage": round(float(col.notna().mean()), 4),
                     "n_nan_file": int(s[tag].isna().sum()),
                     "min": round(float(col.min()), 4), "max": round(float(col.max()), 4),
                     "in_[0,1]": bool(col.dropna().between(0, 1).all())})
    frames[sp] = f
    MODELS[sp] = list(FILES[sp])
inv = pd.DataFrame(rows).set_index(["split", "run"])
display(inv)
bad = inv[(inv["coverage"] < 1.0) | (inv["n_nan_file"] > 0) | (~inv["in_[0,1]"])]
if len(bad):
    print("!! integrity flags above — resolve before reading anything downstream:")
    display(bad)

In [ ]:
# §3 — distributions: summary quantiles + ECDF overlay per split
qs = [0.5, 0.9, 0.99]
rows = []
for sp in SPLITS:
    for tag in MODELS[sp]:
        s = frames[sp][tag].dropna()
        rows.append({"split": sp, "run": tag, "n": len(s),
                     "mean": float(s.mean()), "std": float(s.std()),
                     **{f"p{int(q * 100)}": float(s.quantile(q)) for q in qs},
                     "max": float(s.max())})
display(pd.DataFrame(rows).set_index(["split", "run"]).round(4))

fig, axes = plt.subplots(1, len(SPLITS), figsize=figstyle.FIG_2, sharey=True)
axes = np.atleast_1d(axes)
for ax, sp in zip(axes, SPLITS):
    for tag in MODELS[sp]:
        s = np.sort(frames[sp][tag].dropna().to_numpy(dtype=float))
        ecdf = np.arange(1, len(s) + 1) / len(s)
        if tag == "baseline":
            ax.plot(s, ecdf, ls=":", color=figstyle.NEUTRAL, label="baseline")
        else:
            ax.plot(s, ecdf, label=tag)          # colour cycle assigns slots in order
    for t in TAUS:
        ax.axvline(t, ls="--", lw=1, color=figstyle.INK)
        ax.text(t, 0.02, f" τ={t}", fontsize=8, color=figstyle.INK, rotation=90)
    ax.set_title(f"{VERSION} · {sp}")
    ax.set_xlabel("score")
axes[0].set_ylabel("ECDF")
axes[-1].legend(fontsize=8)
figstyle.save(fig, f"{VERSION}_0304_score_ecdf")
plt.show()

In [ ]:
# §4 — cutoff mass: share above each reference cutoff + boundary-band occupancy
# Label-free would-scrap volume proxy: an axis that shrinks the mass above τ relative to the
# baseline would scrap fewer cars at the production cutoff (its own tuned τ is 03_05's job).
rows = []
for sp in SPLITS:
    for tag in MODELS[sp]:
        s = frames[sp][tag].dropna().to_numpy(dtype=float)
        r = {"split": sp, "run": tag}
        for t in TAUS:
            r[f"share>τ{t}"] = float((s > t).mean())
            r[f"band(τ-{BAND_H},τ]{t}"] = float(((s > t - BAND_H) & (s <= t)).mean())
        rows.append(r)
display(pd.DataFrame(rows).set_index(["split", "run"]).round(5))

In [ ]:
# §5 — agreement: how far each axis moved (vs baseline and pairwise)
for sp in SPLITS:
    f = frames[sp]
    runs = MODELS[sp]
    print(f"\n{VERSION} · {sp} — Spearman rank correlation between runs (common scored rows):")
    display(f[runs].corr(method="spearman").round(4))
    ref = "baseline" if "baseline" in runs else runs[0]
    rows = []
    for tag in runs:
        if tag == ref:
            continue
        both = f[[ref, tag]].dropna()
        delta = (both[tag] - both[ref]).to_numpy(dtype=float)
        rows.append({"run": tag, "vs": ref, "n": len(both),
                     "mean_delta": float(delta.mean()),
                     "mean_abs_delta": float(np.abs(delta).mean()),
                     "max_abs_delta": float(np.abs(delta).max())})
    if rows:
        display(pd.DataFrame(rows).set_index("run").round(4))

## Notes

- `mean_delta < 0` vs baseline means the axis lowered scores on average — with the cutoff-mass
  table (§4) this is the label-free trace of the correction pulling would-scrap volume down.
  Whether that trade is *worth it* (precision/recall at an operating point) is **03_05**.
- naive should sit closest to baseline (same contaminated target, weight 1 everywhere; residual
  gaps come from library/refit noise, and the naive-vs-baseline row is exactly that noise
  floor). transport/pnu should move the most.
- An integrity flag in §2 (coverage < 1, NaNs, out-of-range) invalidates every later table for
  that run — fix the export/scoring first.